In [3]:
import pandas as pd
import seaborn as sns
import numpy as np
from scipy import stats
import matplotlib.pyplot as plt
import plotly.offline as pyo
import plotly.graph_objs as go
import plotly.express as px
from sklearn.preprocessing import RobustScaler, StandardScaler, MinMaxScaler
from sklearn.decomposition import PCA
from scipy.stats.mstats import winsorize
from sklearn.feature_selection import VarianceThreshold
import random
import tensorflow as tf

# Set seed for reproducibility
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
tf.random.set_seed(SEED)

# Load dataset
df = pd.read_csv("../Harga Bahan Pangan/train/Minyak Goreng Curah.csv")

def mape(y_true: np.ndarray, y_pred: np.ndarray) -> float:
    return np.mean(np.abs((y_true - y_pred) / y_true)) * 100

missing_percent_df = df.isnull().sum() / len(df) * 100
print(missing_percent_df)

numeric_features = df.select_dtypes(include=['number']).columns
df[numeric_features] = df[numeric_features].interpolate(method="linear")

def df_to_X_y(df, window_size=5):
    X, y = [], []
    for i in range(len(df) - window_size):
        X.append(df[i:i+window_size])
        y.append(df[i + window_size])
    return np.array(X), np.array(y)

# Preprocess data
df = df.select_dtypes(include=[np.number])
df = df.apply(pd.to_numeric, errors='coerce').dropna()

if df.shape[1] > 1:
    print(f"Warning: DataFrame has multiple numeric columns ({df.shape[1]}). Using the first column.")
    df = df.iloc[:, 0]

# StandardScaler
scaler = StandardScaler()
df = scaler.fit_transform(df.values.reshape(-1, 1))

window_size = 5
X, y = df_to_X_y(df, window_size)

# Train-test split with fixed seed for reproducibility
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, shuffle=False, random_state=SEED)

# Callbacks
checkpoint_path = "model_checkpoint.keras"
cp4 = tf.keras.callbacks.ModelCheckpoint(
    filepath=checkpoint_path,
    save_best_only=True,
    monitor='val_loss',
    mode='min'
)

lr_reducer = tf.keras.callbacks.ReduceLROnPlateau(
    monitor='val_loss', 
    factor=0.8, 
    patience=5, 
    min_lr=1e-6, 
    verbose=1
)

early_stopping = tf.keras.callbacks.EarlyStopping(
    monitor='val_loss',
    patience=10,  
    restore_best_weights=True,
    verbose=1
)

def create_lstm_model(input_shape):
    model = tf.keras.Sequential([
        tf.keras.layers.InputLayer(input_shape=input_shape),
        tf.keras.layers.LSTM(128, return_sequences=True),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(64, return_sequences=True),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.LSTM(32, return_sequences=False),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(16, activation='relu'),
        tf.keras.layers.Dense(1, activation='linear')
    ])
    model.compile(loss='mape', optimizer=tf.keras.optimizers.Adam(learning_rate=0.003), metrics=['mape'])
    return model

input_shape = (X_train.shape[1], X_train.shape[2])
model = create_lstm_model(input_shape)

history = model.fit(X_train, y_train, validation_data=(X_test, y_test), 
          epochs=100, batch_size=32, verbose=1, 
          callbacks=[cp4, lr_reducer, early_stopping])

print("Training completed. Final epoch:", len(history.history['loss']))

model.save("lstm_model.keras")

# Sample submission
df_submission = pd.read_csv("sample_submission.csv")
unique_countries = df_submission['id'].str.split('/').str[1].unique()
total_required_predictions = 92 * len(unique_countries)
print("Total required predictions:", total_required_predictions)
print("Unique countries:", len(unique_countries))

future_predictions = []
input_seq = X_test[-1]

for _ in range(total_required_predictions):
    pred = model.predict(input_seq.reshape(1, window_size, 1))[0, 0]
    pred += np.random.normal(0, 0.01)  
    future_predictions.append(pred)
    input_seq = np.roll(input_seq, -1)
    input_seq[-1] = pred

if len(future_predictions) != total_required_predictions:
    print(f"Warning: Expected {total_required_predictions} predictions but got {len(future_predictions)}")

# Convert predictions back to original scale
y_pred = scaler.inverse_transform(np.array(future_predictions).reshape(-1, 1))

data = []
for i in range(min(len(df_submission), len(y_pred))):
    data.append({'id': df_submission.iloc[i]['id'], 'price': y_pred[i][0]})

submission_df = pd.DataFrame(data)
submission_df.to_csv("minyak_goreng_curah_random_submission_new.csv", index=False)
print("Submission file saved as minyak_goreng_curah_random_submission_new.csv")

Date                          0.000000
Aceh                         14.641434
Bali                         14.541833
Banten                       14.641434
Bengkulu                     14.741036
DI Yogyakarta                14.541833
DKI Jakarta                  14.741036
Gorontalo                    19.223108
Jambi                        14.741036
Jawa Barat                   14.641434
Jawa Tengah                  14.342629
Jawa Timur                   14.442231
Kalimantan Barat             14.541833
Kalimantan Selatan           14.641434
Kalimantan Tengah            14.541833
Kalimantan Timur             14.940239
Kalimantan Utara             29.681275
Kepulauan Bangka Belitung    14.940239
Kepulauan Riau               14.840637
Lampung                      14.641434
Maluku Utara                 34.262948
Maluku                       14.641434
Nusa Tenggara Barat          14.641434
Nusa Tenggara Timur          14.641434
Papua Barat                  14.741036
Papua                    

d:\Anaconda\Lib\site-packages\keras\src\layers\core\input_layer.py:27: UserWarning:

Argument `input_shape` is deprecated. Use `shape` instead.



Epoch 1/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 10s 109ms/step - loss: 81.6898 - mape: 81.6898 - val_loss: 70.4137 - val_mape: 70.4137 - learning_rate: 0.0030
Epoch 2/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 54.9569 - mape: 54.9569 - val_loss: 85.2622 - val_mape: 85.2622 - learning_rate: 0.0030
Epoch 3/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 22ms/step - loss: 48.8600 - mape: 48.8600 - val_loss: 80.3321 - val_mape: 80.3321 - learning_rate: 0.0030
Epoch 4/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 1s 23ms/step - loss: 52.1588 - mape: 52.1588 - val_loss: 79.2511 - val_mape: 79.2511 - learning_rate: 0.0030
Epoch 5/100
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 20ms/step - loss: 50.2966 - mape: 50.2966 - val_loss: 81.4275 - val_mape: 81.4275 - learning_rate: 0.0030
Epoch 6/100
15/18 ━━━━━━━━━━━━━━━━━━━━ 0s 12ms/step - loss: 47.4698 - mape: 47.4698
Epoch 6: ReduceLROnPlateau reducing learning rate to 0.002400000020861626.
18/18 ━━━━━━━━━━━━━━━━━━━━ 0s 25ms/step - loss: 48.6591 - mape: 48.6591 - val_loss: 75.8297 - val_m